# CMSC 471: Introduction to Artificial Intelligence


# Naive Bayes Classification
## Text Classification of Movie Reviews

## Background Section

We will be using the Scikit-learn (sklearn) Python library to explore Naive Bayes for sentiment analysis of movie reviews from IMDB.

## IMDB Movie Review Dataset

We are using the IMDB Large Movie Dataset curated by Maas, et al. for their paper, "Learning Word Vectors for Sentiment Analysis", published in the 2011 *Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies*. The dataset consists of 50,000 movie reviews from IMDB, evenly divided between test and training sets. Only the most polar reviews for movies have been included. Reviews scoring 4 or less out of 10 are labeled negative, and reviews scoring 7 or more out of 10 are labeled positive. The original dataset can be download from:

https://ai.stanford.edu/~amaas/data/sentiment/


We consolidated the reviews from the test and training sets into single files respectively, with each file having three tab-separated fields for the file ID, review, and label. The labels are {0,1} with 0 for negative and 1 for positive.

## Classifying movie reviews

1. import libraries.
2. load datasets

$
\renewcommand\vec{\mathbf}
$

In [ ]:
!pip install wordcloud

In [ ]:
import numpy as np
import pandas as pd
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
import matplotlib.pyplot as plt

### Explore the dataset

Load the training and testing sets into a Pandas dataframe.

In [ ]:
pd.set_option("max_colwidth",120)
imdb_train = pd.read_csv("imdb_movie_review_train.tsv", names=["ID", "Review", "Label"], delimiter="\t")
imdb_test = pd.read_csv("imdb_movie_review_test.tsv", names=["ID", "Review", "Label"], delimiter="\t")

Explore the first 10 reviews in the training set.

In [ ]:
imdb_train.head(10)

Explore the last 10 reviews in the training set.

In [ ]:
imdb_train.tail(10)

### Word Cloud

We'll build a word cloud from the training set.

First concatenate the reviews into a single document.

In [ ]:
# Concatenate the first 5000 reviews from the training set.
text = " ".join(review for review in imdb_train.loc[:5000,'Review'])

# Count words.
wc = len(text.strip().split(" "))
print(f"There are {wc} words in the first 5000 reviews in the training set.")

Build and display the word cloud.

WARNING: This may take some time.

In [ ]:
# Construct word cloud.
wordcloud = WordCloud(stopwords=STOPWORDS,
                      background_color="white",
                      max_words=1000,
                      width=1600,height=800).generate(text)

# Display word cloud.
plt.figure(figsize=(20,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

Next we'll train a classifier on the data and use it to classify data it has not seen before.

## Preparing Training and Testing sets

The dataset has already been split into training and test sets.

The training and testing reviews are the known data X and will be loaded into X_train and X_test, respectively.

Each review has a label y and will be loaded into the correspondig y_train and y_test.

In [ ]:
# Training set
X_train = imdb_train['Review']
y_train = imdb_train['Label']

# Testing set
X_test = imdb_test['Review']
y_test = imdb_test['Label']

## Term Frequency Matrix (Bag of Words)

We need the counts (frequency) of each word (term) as a matrix of term frequencies. Meaning each review will be treated as a bag of words where the ordering of words does not matter.

We will use the following utility in scikit-learn to transform the data into a term frequency matrix.

In [ ]:
tf_vectorizer = CountVectorizer()

## Transform training data into TF matrix

In [ ]:
# Get term frequency and transform to a term frequency matrix.
tf_train = tf_vectorizer.fit_transform(X_train)
print("samples: %d, features: %d" % tf_train.shape)

## Transform testing data into TF matrix

Notice that we use the "transform()" function instead of "fit_transform()" that was used on the training set. This is because we want the vocabulary (set of unique words) and order of terms found from "fit_transform()" to be the same for the test set.

In [ ]:
tf_test = tf_vectorizer.transform(X_test)
print("samples: %d, features: %d" % tf_test.shape)

## Naive Bayes Classifier

We will use the multinomial Naive Bayes classifier.

In [ ]:
classifier = MultinomialNB()

## Training

Train the Naive Bayes classifier on the training set.

In [ ]:
classifier.fit(tf_train, y_train)

## Testing

In [ ]:
y_prediction = classifier.predict(tf_test)

## Accuracy

In [ ]:
score = metrics.accuracy_score(y_test, y_prediction)
print("Accuracy: %0.3f" % score)

### Performance metrics report

In [ ]:
report = metrics.classification_report(y_test, y_prediction)
print(report)

Now plot the confusion matrix and print the accuracy again.

In [ ]:
# Confusion matrix.
_, axes = plt.subplots(figsize=(8, 6))
metrics.plot_confusion_matrix(classifier, tf_test, y_test, ax=axes, normalize='true')
plt.show()
print("Accuracy:", classifier.score(tf_test, y_test))

## Precision and Recall
The classification report has three main statistics that should be inspected: i) precision ii) recall iii) f1-score.

The definitions for precision and recall are defined next.

<b>Precision</b>: Ratio of correctly predicted positives to all positive predictions.

\begin{align*}
\frac{TP}{TP+FP}
\end{align*}

<b>Recall</b>: Ratio of correctly predicted positives to actual positives.

\begin{align*}
\frac{TP}{TP+FN} = \frac{TP}{\mathscr{P}}
\end{align*}

The precision indicates how well the model performs with respect to false positives. If the precision is high then the false positive rate is low.

The recall is the percentage of positives correctly identified. High recall indicates fewer false negatives.

The f1 score measures the accuracy of the model. Mathematically it is the harmonic mean of the precision and recall.

\begin{align*}
\large
F_1 &= \frac{2}{\frac{1}{\text{precision}} + \frac{1}{\text{recall}}}
= \frac{2}{\frac{\text{precision}+\text{recall}}{\text{precision}\times \text{recall}}} \\
&= 2\frac{\text{precision}\times \text{recall}}{\text{precision}+\text{recall}}
\end{align*}

If the precision and recall are perfect then the $F_1=1$. In the worst case $F_1=0$ where the precision or recall is zero.

## Exercise

Improve the accuracy. Print out your new accuracy score.

In [ ]:
# Load data.
imdb_train = pd.read_csv("imdb_movie_review_train.tsv", names=["ID", "Review", "Label"], delimiter="\t")
imdb_test = pd.read_csv("imdb_movie_review_test.tsv", names=["ID", "Review", "Label"], delimiter="\t")

# Remove HTML tags from data.
imdb_train['Review'] = imdb_train['Review'].replace(r'<[^<]+?>', '', regex=True)
imdb_test['Review'] = imdb_test['Review'].replace(r'<[^<]+?>', '', regex=True)


# Training set.
X_train = imdb_train['Review']
y_train = imdb_train['Label']

# Testing set.
X_test = imdb_test['Review']
y_test = imdb_test['Label']


# Get term frequency and transform to a term frequency matrix.
#tf_vectorizer = CountVectorizer(stop_words='english',strip_accents='unicode')

tf_vectorizer = TfidfVectorizer(norm = 'l1', strip_accents='ascii', sublinear_tf = True, max_df= .5, lowercase=False, smooth_idf=False)
tf_train = tf_vectorizer.fit_transform(X_train)
tf_test = tf_vectorizer.transform(X_test)
print("Training samples: %d, features: %d" % tf_train.shape)
print("Testing  samples: %d, features: %d" % tf_test.shape)

# Get classifier and train.
classifier = MultinomialNB(fit_prior = False )

print(classifier.get_params(deep=True))

# Fit and Predict.
classifier.fit(tf_train, y_train)
y_prediction = classifier.predict(tf_test)

# Accuracy.
score = metrics.accuracy_score(y_test, y_prediction)
print("Accuracy: %0.3f" % score)

## End of Exercise
## CMSC 471